In [1]:
%pip install python-dotenv openai datasets math_verify tqdm torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import os
from openai import AsyncOpenAI
from math_verify import parse

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=256,
)

prompt = "What is 13 times 17? Box your answer."
gold = "221"

completion = await client.chat.completions.create(
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(completion.choices):
  message = choice.message
  parsed_answer = parse(message.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Generation {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(message.reasoning_content + message.content)

******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Generation 1: 221 (correct) ********************
Here's a thinking process:

1.  **Identify the user's request**: The user wants to know the product of 13 and 17, and they want the answer boxed.

2.  **Calculate 13 × 17**:
   - I can do this mentally or write it out.
   - 13 × 17 = 13 × (10 + 7) = 130 + 91 = 221
   - Alternatively: 17 × 10 = 170, 17 × 3 = 51, 170 + 51 = 221
   - Or: 13 × 17 = (15 - 2)(15 + 2) = 15² - 2² = 225 - 4 = 221
   - All methods give 221.

3.  **Format the answer**: The user said "Box your answer." I need to present the final answer within a box. Typically, this means using LaTeX \boxed{221} or just plain text with brackets. Since it's a math question, I'll use \boxed{221}.

4.  **Construct the response**: Just give the boxed answer, maybe with a brief statement, but the instruction is "Box your
******************** Generation 2: 221 (correct) ************

In [4]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [ ]:
from math_verify import verify
import torch
from datasets import Dataset
from tqdm.asyncio import tqdm_asyncio

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"advanatages": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(client.chat.completions.create(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
	for prompt, completion, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		rewards = []
		for choice in completion.choices:
			message = choice.message
			answer = parse(message.content)
			correct = verify(gold, answer)

			outputs.append(f"<think>{message.reasoning_content}</think>{message.content}")
			rewards.append(1.0 if correct else 0.0)
		rewards = torch.tensor(rewards)

		rewards_std = rewards.std()
		if rewards_std < 1e-5:
			advantages = torch.zeros_like(rewards)
		else:
			advantages = (rewards - rewards.mean()) / rewards_std

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["advantages"].append(advantages.tolist())
	return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)
example_ds[:]

2026-08-25 23:05:24,160 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-25 23:06:19,390 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
Creating completions: 100%|██████████| 2/2 [01:14<00:00, 37.15s/it]


{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks to "Pick a random integer from 1 to 3."\n   - Constraint: "Don\'t pick 2."\n   - Output format: "Box your answer."\n\n2.  **Identify the Task:**\n   - Select an integer from {1, 2, 3} excluding 2.\n   - So the valid choices are 1 or 3.\n   - The user says "random," but there\'s a constraint that excludes 2. I need to pick one of the remaining options.\n   - Since it\'s "random" but constrained, I\'ll pick one arbitrarily but fairly from {1, 3}. I can just choose 1 or 3. Let\'s pick 3 (or 1, doesn\'t matter as long as it\'s not 2).\n   - Output must be boxed: e.g., \\boxed{3} or \\boxed{1}.\n\n3.  **Decision:** \n   - I\'ll pick 3. \n   - Format: \\boxed{3}\n\n4.  **Check Constraints:**\n   - From 1 to 3: Yes, 3 is in that range.\n   - Don\'t pick 2: Yes, 3 ≠ 2.\n   - B

In [ ]:
input_ds = ds["train"].shuffle().select(range(128))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15
    },
)
output_ds

2026-08-25 23:06:19,832 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-25 23:06:19,834 [INFO] Retrying request to /chat/completions in 0.435270 seconds
2026-08-25 23:06:19,836 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-25 23:06:19,838 [INFO] Retrying request to /chat/completions in 0.377965 seconds
2026-08-25 23:06:19,839 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-25 23:06:19,841 [INFO] Retrying request to /chat/completions in 0.394400 seconds
2026-08-25 23:06:19,846 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-08-25 23:06:19,848 [INFO] Retrying request to /chat/completions in 0.490520 seconds
2026-08-25 23:06:19,850 [INFO] HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completio